# Sentiment Analysis on IMDB Movie Reviews with Word2Vec + LSTM

Binary sentiment classification (positive/negative) on the [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews).

**Pipeline:** raw review text → cleaned & tokenized → Word2Vec embeddings (trained on the training split only, to avoid data leakage) → embedding matrix → two-layer LSTM → positive/negative prediction.

**Result:** 84.69% accuracy on a 15,000-review held-out test set.

See the [README](README.md) for full write-up, architecture details, and how to run this notebook.

## Setup

In [1]:
!pip install gensim

## Load the Data

Download `IMDB Dataset.csv` from [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) and place it in this folder (not included in this repo, see README).

`data_cleaner.py` (included in this repo) provides `clean_string()`, used below to lowercase, strip HTML tags, and normalize punctuation spacing before tokenizing.

In [1]:
import numpy as np
import pandas as pd
import data_cleaner
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input

#data preprocessing
#read in csv
df_reviews = pd.read_csv("IMDB Dataset.csv", engine='python', on_bad_lines='skip')
print("dataset shape", df_reviews.shape)
print(df_reviews.head())

dataset shape (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


## Clean & Tokenize Reviews

In [1]:
text = df_reviews['review'].values
tokens = [data_cleaner.clean_string(x) for x in text]
tokens = [t.split() for t in tokens if t]
print(tokens[:1])

[['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', ...]]


## Train/Test Split

The raw tokens are split **before** training Word2Vec, so the test reviews never influence the embeddings (avoids data leakage).

In [1]:
# turn +ve/-ve labels into numbers, 1/0
y_binary = (df_reviews['sentiment'] == 'positive').astype(int).values

# split the raw tokens BEFORE training Word2Vec so the
# test reviews never influence the embeddings
tokens_train, tokens_test, y_train_raw, y_test_raw = train_test_split(
    tokens, y_binary, test_size=0.3, random_state=42
)
print(len(tokens_train), len(tokens_test))

35000 15000


## Train Word2Vec Embeddings

Trained on the **training split only** (100-dim vectors, CBOW, window of 2, min count of 5).

In [1]:
model = Word2Vec(
    sentences=tokens_train,  # train reviews only
    vector_size=100,
    window=2,
    sg=0, #CBOW
    min_count=5, #ignore appearing less than 5 times
    sample=0,
    negative=5,
    workers=4,
    epochs=5,
)

print(model.wv.most_similar("good"))

[('great', 0.7714986801147461), ('bad', 0.7585238218307495), ('decent', 0.7491788864135742), ('cool', 0.720865786075592), ('fine', 0.7054243683815002), ('weak', 0.6855128407478333), ('passable', 0.6789767146110535), ('solid', 0.6718106269836426), ('nice', 0.671584963798523), ('funny', 0.6696779727935791)]


In [1]:
print(model.wv.index_to_key[:10]) #most frequent words

['the', ',', 'and', 'a', 'of', 'to', 'is', 'it', 'in', 'i']


In [ ]:
model.save("mymodel.model")

## Build the Embedding Matrix

Keras' `Embedding` layer needs a lookup table indexed by word ID. Row 0 is reserved for padding (left as zeros); every other row is that word's Word2Vec vector.

In [ ]:
#create an embedding matrix to be used in keras as the embedding layer
vocab = model.wv.key_to_index
vocab_size = len(vocab) + 1
embedding_dim = model.wv.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in vocab.items():
    embedding_matrix[idx + 1] = model.wv[word]

In [1]:
word_id = vocab["good"]
print(embedding_matrix[word_id + 1])   # +1 because of the padding shift

[-0.99835515 -1.69235122  3.00353765 -2.04598236 -2.86206031 -1.69095147
  2.21250558  3.38856649  0.17985784 -2.13933015  ...]


## Pad Sequences to a Fixed Length

Every review is converted to a list of word IDs and padded/truncated to exactly 200 tokens so the LSTM receives fixed-shape input.

In [1]:
# 1. Convert the tokenized sequences to padded numerical vectors
sequence_length = 200

def to_padded(tokens):
    sequences = [[vocab[w] + 1 for w in review if w in vocab] for review in tokens]
    return pad_sequences(sequences, maxlen=sequence_length,
                         padding='pre', truncating='post')
    ##put zeros at the front,truncate long reviews at the end

X_train, X_test = to_padded(tokens_train), to_padded(tokens_test)
y_train = to_categorical(y_train_raw, num_classes=2)
y_test = to_categorical(y_test_raw, num_classes=2)
print(X_train.shape, X_test.shape)

(35000, 200) (15000, 200)


In [1]:
print(len(df_reviews), len(tokens), len(vocab), X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(df_reviews['sentiment'].value_counts())

50000 50000 34736 (35000, 200) (15000, 200) (35000, 2) (15000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


## Build the Model

Two stacked LSTM layers (64 units each). Embeddings are **unfrozen** (`trainable=True`), so the Word2Vec vectors continue to adapt during training specifically for sentiment classification.

In [1]:
# model with two stacked LSTMs of 64
vocab_size, embedding_dim = embedding_matrix.shape
lstm_model = Sequential([
    Input(shape=(sequence_length,)),
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=True,
    ),
    LSTM(64, return_sequences=True),  # passes all 200 steps to the next LSTM
    LSTM(64),                          # keeps only the final step
    Dropout(0.15),
    Dense(2, activation='softmax'),
])

lstm_model.compile(
    optimizer='rmsprop', #adjusts model weights
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
lstm_model.summary()

Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 100)       │     3,473,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 200, 64)        │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           130 │
└─────────────────────────────────┴─────────────────────

## Train & Evaluate

In [1]:
# train with training set, then test with held out testing set
history = lstm_model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
)

test_loss, test_acc = lstm_model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1/5
493/493 - 113s 226ms/step - accuracy: 0.7147 - loss: 0.5612 - val_accuracy: 0.7677 - val_loss: 0.4836
Epoch 2/5
493/493 - 112s 228ms/step - accuracy: 0.7904 - loss: 0.4622 - val_accuracy: 0.7660 - val_loss: 0.5183
Epoch 3/5
493/493 - 107s 218ms/step - accuracy: 0.8244 - loss: 0.4043 - val_accuracy: 0.8423 - val_loss: 0.3646
Epoch 4/5
493/493 - 106s 215ms/step - accuracy: 0.8471 - loss: 0.3595 - val_accuracy: 0.8131 - val_loss: 0.4656
Epoch 5/5
493/493 - 107s 217ms/step - accuracy: 0.8656 - loss: 0.3279 - val_accuracy: 0.8411 - val_loss: 0.3465
469/469 - 16s 34ms/step - accuracy: 0.8469 - loss: 0.3486
Test accuracy: 0.8469


## Try It on New Reviews

A quick sanity check on reviews the model has never seen, including a deliberately ambiguous/neutral one.

In [1]:
##random reviews i made up to test the model
reviews = [
    "This movie was a complete waste of time, terrible acting and a boring plot.",
    "Ehh, I thought it was just alright, nothing special",
    "I absolutely loved it!",
    "This movie was good.",
    "This movie was bad."
]

# Clean and tokenise
cleaned_reviews = [data_cleaner.clean_string(r).split() for r in reviews]

# clean reviews into word IDs using the existing vocab dictionary
sequences = [[vocab[w] + 1 for w in review if w in vocab] for review in cleaned_reviews]
padded = pad_sequences(sequences, maxlen=sequence_length, padding='pre', truncating='post')
predictions = lstm_model.predict(padded)

labels = ["Negative", "Positive"]
for review, pred in zip(reviews, predictions):
    print(f"{review}\n  \u2192 {labels[pred.argmax()]} "
          f"(P(positive) = {pred[1]:.2f})")

This movie was a complete waste of time, terrible acting and a boring plot.
  → Negative (P(positive) = 0.01)
Ehh, I thought it was just alright, nothing special
  → Negative (P(positive) = 0.12)
I absolutely loved it!
  → Positive (P(positive) = 0.98)
This movie was good.
  → Positive (P(positive) = 0.68)
This movie was bad.
  → Negative (P(positive) = 0.16)
